In [40]:
import torch
import torch.nn as nn
from torchvision.datasets import CIFAR10
from torch.utils.data import Dataset,DataLoader
import torch.optim as optim
from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)
from torchvision.models import vgg19,VGG19_Weights
import torchvision.transforms as T

In [41]:
device = torch.device('mps' if torch.mps.is_available else 'cpu')
device

device(type='mps')

In [42]:
model = vgg19(weights=VGG19_Weights.DEFAULT)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [43]:
for name,param in model.named_parameters():
    if not name.startswith('classifier'):
       param.requires_grad = False

In [44]:
for param in model.parameters():
    print(param.requires_grad)

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
True
True
True
True
True


In [45]:
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225])]
)
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(in_features=num_features,out_features=len(test_dataset.classes))

In [46]:
num_features = model.classifier[6].in_features
num_features

4096

In [47]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(),lr=1e-3)

In [48]:
model.to(device)
from tqdm import tqdm
def train_loop(model,train_loader,criterion,optimizer):
    model.train()
    run_loss,correct,total = 0,0,0

    for imgs,lbls in tqdm(train_loader):
        imgs = imgs.to(device)
        lbls = lbls.to(device)
        logits = model(imgs)
        loss = criterion(logits,lbls)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        run_loss += loss.item()
        preds = torch.argmax(logits,dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

    loss , acc = run_loss/len(train_loader) , correct/total
    print(f"Loss -> {loss:.4f}, Acc -> {acc:.4f}")

In [49]:
@torch.no_grad()
def test_loop(model,test_loader,criterion,optimizer):
    correct , total = 0,0
    model.eval()
    for imgs,lbls in tqdm(test_loader):
        imgs = imgs.to(device)
        lbls = lbls.to(device)
        logits = model(imgs)
        preds = torch.argmax(logits,dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)
    print(f"ACC : {correct/total}")

In [50]:
for epoch in range(2):
    print(f'Epoch: {epoch+1}')
    train_loop(model=model,train_loader=train_loader,criterion=criterion,optimizer=optimizer)

Epoch: 1


  6%|██▎                                     | 92/1563 [00:49<13:12,  1.86it/s]


KeyboardInterrupt: 

In [35]:
test_loop(model=model,test_loader=test_loader,criterion=criterion,optimizer=optimizer)

100%|████████████████████████████████████████| 313/313 [02:29<00:00,  2.10it/s]

ACC : 0.474


In [36]:
for params in model.parameters():
    params.requires_grad = True

In [37]:
for epoch in range(15):
    print(f'Epoch: {epoch+1}')
    train_loop(model=model,train_loader=train_loader,criterion=criterion,optimizer=optimizer)

Epoch: 1


100%|████████████████████████████████████| 1563/1563 [1:05:06<00:00,  2.50s/it]


Loss -> 2.4309, Acc -> 0.4205
Epoch: 2


100%|████████████████████████████████████| 1563/1563 [1:11:04<00:00,  2.73s/it]


Loss -> 2.3244, Acc -> 0.4444
Epoch: 3


...

Loss -> 2.1789, Acc -> 0.4836
Epoch: 5


100%|████████████████████████████████████| 1563/1563 [1:26:45<00:00,  3.33s/it]


Loss -> 2.1548, Acc -> 0.4899
Epoch: 6


 59%|███████████████████████                | 926/1563 [51:02<35:06,  3.31s/it]


KeyboardInterrupt: 

In [38]:
print(device)
print(next(model.parameters()).device)

mps
mps:0
